# CSC5035Z A2 — Experiment Pipeline
**Student:** Praise Jaravani (JRVPRA001) | **Language:** Yoruba (`yor`)

This notebook runs the complete experiment pipeline. It is structured in two passes:

| Pass | Steps | LR | Purpose |
|---|---|---|---|
| Initial run | 1–8 | 2e-5 | Establish baseline and Extension B |
| Sweep + retrain | 9–10 | 5e-5 | Identify optimal LR, retrain all models |

**The outputs from Step 10 are the final reported results.**

---

### Scripts in this project
| Script | Purpose |
|---|---|
| `config.py` | Single source of truth for all hyperparameters and paths |
| `utils.py` | Shared helpers: seed setting, LR scheduler, NER label alignment |
| `analyse_tokenizer.py` | Measures token-per-word fertility on the Yoruba corpus |
| `train_news.py` / `train_news_extb.py` | Fine-tune mmBERT-small on MasakhaNews (baseline / Ext B) |
| `train_ner.py` / `train_ner_extb.py` | Fine-tune mmBERT-small on MasakhaNER 2.0 (baseline / Ext B) |
| `extend_vocab.py` | Extension B: trains Yoruba BPE vocab, extends tokenizer + model embeddings |
| `sweep_lr.py` | Sweeps LR in {1e-5, 2e-5, 3e-5, 5e-5} on train/val only — test set never touched |
| `evaluate.py` | Standalone test-set evaluation for any saved checkpoint |

**After a Colab disconnect:** re-run Cell 1 only, then resume from where you left off.

In [ ]:
# Cell 1 — Set up working directory (works on Google Colab and locally)
import os, pathlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/csc5035z-a2'
except ImportError:
    # Running locally — assumes Jupyter was launched from the project root
    PROJECT_PATH = str(pathlib.Path().resolve())

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')
print('Files found:', os.listdir('.'))

In [ ]:
# Cell 2 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
print('Install complete' if result.returncode == 0 else result.stderr)

In [ ]:
# Cell 4 — Verify imports
import torch, transformers, datasets
print(f'PyTorch:       {torch.__version__}')
print(f'Transformers:  {transformers.__version__}')
print(f'Datasets:      {datasets.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')

---
## Step 1: Tokenizer fertility analysis — baseline

**Script:** `analyse_tokenizer.py`

Measures how the stock mmBERT-small tokenizer handles Yoruba text by computing the token-per-word ratio across the full training corpus (MasakhaNews + MasakhaNER, 8,309 sentences, ~198k words).

Yoruba uses stacked Unicode diacritics that standard multilingual tokenizers split into individual combining characters, producing very high fertility.

**What to look for in the output:**
- `Mean toks/word` — average tokens per whitespace-split word; higher = worse
- `% single-token` — fraction of words tokenised as one unit; lower = worse
- `Diacritic word probe` — qualitative check on 10 common Yoruba words with diacritics

**Output:** `results/tokenizer_analysis_baseline.json`

In [ ]:
!python analyse_tokenizer.py

---
## Step 2: Baseline training — MasakhaNews (text classification)

**Script:** `train_news.py`

Fine-tunes mmBERT-small on the MasakhaNews Yoruba subset (1,433 train / 206 val / 411 test, 5 categories). A linear classification head is added over the `[CLS]` token representation.

**Training details:**
- Loss: cross-entropy | Optimiser: AdamW | Scheduler: linear warmup (100 steps)
- Early stopping: patience = 2 epochs on validation macro-F1
- Best checkpoint saved and reloaded for test evaluation — test set never seen during training

**Note on the load report:** `UNEXPECTED` keys are decoder/MLM weights from pre-training (harmless — not needed for classification). `MISSING` keys are the new classification head being randomly initialised (expected).

**Output:** `checkpoints/news_baseline/` and `results/news_baseline.json`

In [ ]:
!python train_news.py

---
## Step 3: Baseline training — MasakhaNER 2.0 (token classification)

**Script:** `train_ner.py`

Fine-tunes mmBERT-small on MasakhaNER 2.0 Yoruba (6,876 train / 983 val / 1,964 test). A linear head is applied to every token representation to predict BIO labels (O, B/I-PER, B/I-ORG, B/I-LOC, B/I-DATE).

**Label alignment:** The first subword of each word receives the word-level NER label; continuation subwords and special tokens receive label `-100`, which is excluded from the cross-entropy loss. This prevents the model from learning inconsistent span boundaries.

**Evaluation:** seqeval span-level F1 — exact span boundaries must match; partial credit is not given.

**Output:** `checkpoints/ner_baseline/` and `results/ner_baseline.json`

In [ ]:
!python train_ner.py

---
## Step 4: Extension B — Vocabulary adaptation

**Script:** `extend_vocab.py`

Extends the mmBERT-small tokenizer with Yoruba-specific BPE tokens to reduce the token-per-word ratio caused by diacritic fragmentation. The pipeline has six steps:

1. **Corpus collection** — combine Yoruba train splits from both tasks (~8k sentences)
2. **BPE training** — HuggingFace `tokenizers` library; `vocab_size=1000`, `min_frequency=5`
3. **Token selection** — diff BPE vocab vs mmBERT vocab; keep tokens with length >= 2
4. **Tokenizer extension** — `add_tokens()` adds new entries; vocab grows 256k → 256,637
5. **Embedding initialisation** — each new token's embedding is set to the mean of the original tokenizer's subword embeddings for that string (principled init; 0 random-only fallbacks)
6. **Verification** — reloads the saved model and prints before/after tokenisation for 10 Yoruba diacritic words

**Output:** `checkpoints/extended_model/` (extended tokenizer + base model weights)

In [ ]:
!python extend_vocab.py

---
## Step 5: Tokenizer fertility analysis — after extension

**Script:** `analyse_tokenizer.py --tokenizer checkpoints/extended_model`

Runs the identical fertility analysis as Step 1 but on the extended tokenizer. Directly quantifies the improvement from vocabulary adaptation.

**What to look for:**
- Drop in `Mean toks/word` compared to Step 1 baseline
- Rise in `% single-token` words
- Diacritic probe: words like *ọmọ* and *ọjọ* should now appear as single tokens

**Output:** `results/tokenizer_analysis_extb.json`

In [ ]:
!python analyse_tokenizer.py --tokenizer checkpoints/extended_model

---
## Step 6: Extension B training — MasakhaNews

**Script:** `train_news_extb.py`

Identical training procedure to Step 2 but uses the extended tokenizer and model from `checkpoints/extended_model/`. Same hyperparameters as the baseline to ensure a fair comparison — no re-tuning for the extended model.

The embedding matrix already has 256,637 rows with principled initialisations for the 637 new tokens. Only the classification head is freshly initialised.

**Output:** `checkpoints/news_extb/` and `results/news_extb.json`

In [ ]:
!python train_news_extb.py

---
## Step 7: Extension B training — MasakhaNER 2.0

**Script:** `train_ner_extb.py`

Identical training procedure to Step 3 but uses the extended vocabulary model. Lower tokenizer fertility means more complete sentences fit within the 128-token window, so the model may be evaluated on a slightly larger number of entities than the baseline (longer sentences are no longer truncated).

**Output:** `checkpoints/ner_extb/` and `results/ner_extb.json`

In [ ]:
!python train_ner_extb.py

---
## Step 8: Test-set evaluation — initial run (LR = 2e-5)

**Script:** `evaluate.py`

Runs all four saved checkpoints on their respective held-out test sets. This is a read-only evaluation step — no model weights are updated.

> **Note:** These results are from the initial LR = 2e-5 run. Step 9 (LR sweep) subsequently identified 5e-5 as the optimal learning rate, and Step 10 retrains all models accordingly. **The Step 10 outputs are the final reported figures.**

In [ ]:
!python evaluate.py --checkpoint checkpoints/news_baseline --task news
!python evaluate.py --checkpoint checkpoints/ner_baseline  --task ner
!python evaluate.py --checkpoint checkpoints/news_extb     --task news
!python evaluate.py --checkpoint checkpoints/ner_extb      --task ner

In [ ]:
# Print results summary
import json, os, glob

print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

for f in sorted(glob.glob('results/*.json')):
    name = os.path.basename(f).replace('.json', '')
    with open(f) as fp:
        data = json.load(fp)
    print(f'\n{name}:')
    for k, v in data.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
        elif k not in ('config', 'timestamp', 'test_predictions', 'test_true_labels',
                       'test_pred_tags', 'test_true_tags', 'per_class_f1', 'per_entity_f1'):
            print(f'  {k}: {v}')

---
## Step 9: Learning rate sweep

**Script:** `sweep_lr.py`

Sweeps four LR candidates — {1e-5, 2e-5, 3e-5, 5e-5} — on both tasks using **train and validation splits only**. The test set is never seen during the sweep.

Each trial trains for up to 5 epochs with early stopping (patience = 2) and reports the best validation F1. No checkpoints are saved — this is a hyperparameter selection step only.

**Result:** LR = 5e-5 achieved the best validation F1 on both tasks, outperforming the 2e-5 used in Steps 1–8. All models are retrained in Step 10.

**Output:** `results/lr_sweep.json`

In [ ]:
!python sweep_lr.py

---
## Step 10: Retrain all models with optimal LR (5e-5)

The sweep identified **LR = 5e-5** as best for both tasks:

| LR | News val-F1 | NER val-F1 |
|---|---|---|
| 1e-5 | 0.7983 | 0.7446 |
| 2e-5 | 0.8414 | 0.7791 |
| 3e-5 | 0.8293 | 0.8198 |
| **5e-5** | **0.8533** | **0.8198** |

`config.py` was updated to `LEARNING_RATE = 5e-5`. All four models are retrained from scratch and re-evaluated on the test set. These are the final reported results.

In [ ]:
!python train_news.py

In [ ]:
!python train_ner.py

In [ ]:
!python train_news_extb.py

In [ ]:
!python train_ner_extb.py

In [ ]:
!python evaluate.py --checkpoint checkpoints/news_baseline --task news
!python evaluate.py --checkpoint checkpoints/ner_baseline  --task ner
!python evaluate.py --checkpoint checkpoints/news_extb     --task news
!python evaluate.py --checkpoint checkpoints/ner_extb      --task ner

---
## Experiment complete

All results saved to `results/`. Open `notebooks/results_analysis.ipynb` to generate summary tables, figures, and error analysis.

### Final results (LR = 5e-5)
| Model | MasakhaNews Macro-F1 | MasakhaNER Span-F1 |
|---|---|---|
| mmBERT-small Baseline | **0.8641** | **0.8407** |
| mmBERT-small + Vocab Ext (Ext B) | 0.8605 | 0.8393 |
| Delta | −0.0036 | −0.0014 |

**Key finding:** Vocabulary adaptation reduced tokenizer fertility by 47% (3.12 → 1.66 mean tokens/word) but had negligible downstream effect at the optimal LR. The NER regression seen at 2e-5 was a learning rate artefact.

#### Extension B: PEFT method comparison
This section below aims to compare various PEFT methods against the full fine-tuning results. Findings of analysis are stored in results directory.

In [ ]:
# 1. Run LoRA sweeps (r=8, r=16, r=32``)
!python peft_comparison.py --method lora --r 8
!python peft_comparison.py --method lora --r 16
!python peft_comparison.py --method lora --r 32

# 2. Run IA3 adapter
!python peft_comparison.py --method ia3

# 3. Run Prompt Tuning sweeps (v=10, v=20, v=30)
!python peft_comparison.py --method prompt --num_virtual_tokens 10
!python peft_comparison.py --method prompt --num_virtual_tokens 20
!python peft_comparison.py --method prompt --num_virtual_tokens 30